### Decorator based middleware

In [ ]:
from langchain.agents.middleware import (
    before_model,
    before_agent,
    after_model,
    after_agent,
    wrap_model_call,
    AgentState,
    
    ModelRequest,
    ModelResponse,
)
from langchain.agents import create_agent
from langgraph.runtime import Runtime
from typing import Any, Callable

@before_agent
def log_before_agent(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(f"Before agent called with {len(state['messages'])} messages")
    return None

@wrap_model_call
def retry_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    print("Model call with retry middleware")
    for attempt in range(3):
        try:
            return handler(request)
        except Exception as e:
            if attempt == 2:
                raise
            print(f"Retry {attempt + 1}/3 after error: {e}")

@before_model
def log_before_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(f"Before model called with {len(state['messages'])} messages")
    return None

@after_model
def log_after_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(f"After model called with {len(state['messages'])} messages")
    return None

@after_agent
def log_after_agent(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(f"After agent called with {len(state['messages'])} messages")
    return None

#     request: ModelRequest,
#     handler: Callable[[ModelRequest], ModelResponse],
# ) -> ModelResponse:
#     for attempt in range(3):
#         try:
#             return handler(request)
#         except Exception as e:
#             if attempt == 2:
#                 raise
#             print(f"Retry {attempt + 1}/3 after error: {e}")



In [56]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

from models.llm import get_llm
from tools.weather_tool import weather_tool
from tools.email_tool import send_email_tool, read_email_tool


agent =  create_agent(
        model = get_llm(),
        tools = [weather_tool, send_email_tool, read_email_tool],
        checkpointer=InMemorySaver(),
        middleware=[
            log_before_agent,
            log_before_model,
            log_after_model,
            log_after_agent,
            retry_model,
        ]
    )


config = {
    "configurable":{
        "thread_id": "test1"
    }
}

In [57]:
response = agent.invoke({"messages":[
    ("human", "Hi I am Hamza"),

]},config)
response

Before agent called with 1 messages
Before model called with 1 messages
Model call with retry middleware
After model called with 2 messages
After agent called with 2 messages


{'messages': [HumanMessage(content='Hi I am Hamza', additional_kwargs={}, response_metadata={}, id='ceb07b1a-781b-42c6-a1a1-8af12ae7ba36'),
  AIMessage(content='Hello Hamza! How can I assist you today?', additional_kwargs={'reasoning_content': "Okay, the user introduced himself as Hamza. I need to respond politely. Let me check if there's any function I need to call here. The tools provided are for weather, sending emails, and reading emails. Since the user just said hi, there's no need to use any function right now. I should keep it simple and friendly. Maybe say hello back and ask how I can assist him. No tool calls required here.\n"}, response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 294, 'total_tokens': 398, 'completion_time': 0.187890813, 'completion_tokens_details': {'reasoning_tokens': 88}, 'prompt_time': 0.015428772, 'prompt_tokens_details': None, 'queue_time': 0.007198354, 'total_time': 0.203319585}, 'model_name': 'qwen/qwen3-32b', 'system_fingerpri

In [25]:
response = agent.invoke({"messages":[
    ("human", "Hi what is my name?"),

]},config)
response

Before agent called with 3 messages
Before model called with 3 messages
After model called with 4 messages
After agent called with 4 messages


{'messages': [HumanMessage(content='Hi I am Hamza', additional_kwargs={}, response_metadata={}, id='f8e2ac0b-691c-4caa-95d7-e09cf710e359'),
  AIMessage(content='Hello Hamza! How can I assist you today?', additional_kwargs={'reasoning_content': "Okay, the user introduced himself as Hamza. I need to respond politely. Let me check if there's any function I need to call here. The tools provided are for weather, sending emails, and reading emails. Since he just said hi, there's no need to use any functions right now. I'll just greet him back and offer assistance.\n"}, response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 294, 'total_tokens': 381, 'completion_time': 0.165318967, 'completion_tokens_details': {'reasoning_tokens': 71}, 'prompt_time': 0.012042645, 'prompt_tokens_details': None, 'queue_time': 0.033555941, 'total_time': 0.177361612}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'stop', '

In [10]:
questions = [
    "Keep all these answers short and in one line",
    "Hi I am Hamza",
    "what is my name?",
    "what is python?",
    "what is compiled vs interpreted language?",
    "what is static vs dynamic language?",
]

for question in questions:
    response = agent.invoke({"messages":[
        ("human", question),
    ]},config)
    print(f"Message: {response['messages'][-1].content}")
    print(f"Count: {len(response['messages'])}")

    

About to call model with 1 messages
Message: Understood, I'll keep answers short and to the point.
Count: 2
About to call model with 3 messages
Message: Hello, Hamza!
Count: 4
About to call model with 5 messages
Message: Your name is Hamza.
Count: 6
About to call model with 7 messages
Message: Python is a high-level, interpreted programming language known for its readability and simplicity, widely used in web development, data analysis, artificial intelligence, and more.
Count: 8
About to call model with 9 messages
Message: Compiled languages (e.g., C) are translated into machine code before execution, while interpreted languages (e.g., Python) are executed line-by-line during runtime.
Count: 10
About to call model with 11 messages
Message: Static languages (e.g., Java) enforce type checking at compile time, while dynamic languages (e.g., Python) resolve types during runtime.
Count: 12


### Class based middleware

In [ ]:
from langchain.agents.middleware import AgentMiddleware

class LoggingMiddleware(AgentMiddleware):
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        print(f"Before agent called with {len(state['messages'])} messages")
        return None

    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        print(f"Before model called with {len(state['messages'])} messages")
        return None

    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        print(f"After model called with {len(state['messages'])} messages")
        return None

    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        print(f"After agent called with {len(state['messages'])} messages")
        return None



In [50]:
agent = create_agent(
    model=get_llm(),
    tools=[weather_tool, send_email_tool, read_email_tool],
    middleware=[LoggingMiddleware()],
)


In [51]:
response = agent.invoke({"messages":[
    ("human", "Hi I am Hamza"),

]})
response

Before agent called with 1 messages
Before model called with 1 messages
After model called with 2 messages
After agent called with 2 messages


{'messages': [HumanMessage(content='Hi I am Hamza', additional_kwargs={}, response_metadata={}, id='aa1796fe-4bba-4376-a631-351a53e2d4c7'),
  AIMessage(content='Hello Hamza! How can I assist you today?', additional_kwargs={'reasoning_content': "Okay, the user introduced himself as Hamza. I need to respond politely. Let me check if there's any function I need to call here. The tools provided are for weather, sending emails, and reading emails. Since the user just said hi, there's no need to use any function right now. I should keep it simple and friendly. Maybe say hello back and ask how I can assist him. No tool calls required here.\n"}, response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 294, 'total_tokens': 398, 'completion_time': 0.187788266, 'completion_tokens_details': {'reasoning_tokens': 88}, 'prompt_time': 0.015085786, 'prompt_tokens_details': None, 'queue_time': 0.397458126, 'total_time': 0.202874052}, 'model_name': 'qwen/qwen3-32b', 'system_fingerpri

In [36]:
response = agent.invoke({"messages":[
    ("human", "Hi what is my name?"),

]},config)
response

Before agent called with 1 messages
Before model called with 1 messages
After model called with 2 messages
After agent called with 2 messages


{'messages': [HumanMessage(content='Hi what is my name?', additional_kwargs={}, response_metadata={}, id='bef139bc-ed38-4d94-9832-8a5601e45b0c'),
  AIMessage(content="I don't have access to your personal information, including your name. However, if you'd like help with something else (like checking the weather, sending an email, etc.), I can assist with that! Let me know how I can help.", additional_kwargs={'reasoning_content': 'Okay, the user is asking, "Hi what is my name?" Let me think about how to handle this.\n\nFirst, I need to check if any of the provided tools can help answer this. The available tools are weather_tool, send_email_tool, and read_email_tool. \n\nThe weather tool gets the current weather based on location. That doesn\'t seem relevant here. The send_email tool allows sending emails, and read_email tool lets you read an email by ID. But the user isn\'t providing an email ID or mentioning sending an email. They\'re asking about their name.\n\nSince the user\'s quest

Before agent called with 1 messages
Before model called with 1 messages
After model called with 2 messages
After agent called with 2 messages
Message: Understood, I'll keep answers short and to the point.
Count: 2
Before agent called with 1 messages
Before model called with 1 messages
After model called with 2 messages
After agent called with 2 messages
Message: Hello Hamza! How can I assist you today?
Count: 2
Before agent called with 1 messages
Before model called with 1 messages
After model called with 2 messages
After agent called with 2 messages
Message: I don't have access to your personal information, including your name. How would you like me to assist you?
Count: 2
Before agent called with 1 messages
Before model called with 1 messages
After model called with 2 messages
After agent called with 2 messages
Message: The tools provided do not include any functions capable of explaining what Python is. Python is a high-level, interpreted programming language known for its readabili

KeyboardInterrupt: 